# RetailPulse – Inventory Optimization

This notebook performs:
- Inventory analysis
- Reorder recommendations
- Stock risk detection
- Demand-based inventory optimization

IMPORT LIBRARIES

In [1]:
import pandas as pd
import numpy as np

CREATE INVENTORY DATA

In [2]:
inventory_data = {

    'Product': ['A', 'B', 'C', 'D'],

    'CurrentStock': [120, 50, 200, 40],

    'ForecastDemand': [180, 70, 160, 90]
}

inventory_df = pd.DataFrame(
    inventory_data
)

inventory_df

,Product,CurrentStock,ForecastDemand
0,A,120,180
1,B,50,70
2,C,200,160
3,D,40,90


INVENTORY FORMULA

In [3]:
# Safety stock

SAFETY_STOCK = 20

# Reorder quantity

inventory_df['ReorderQuantity'] = (

    inventory_df['ForecastDemand']

    + SAFETY_STOCK

    - inventory_df['CurrentStock']
)

# Remove negative values

inventory_df['ReorderQuantity'] = (

    inventory_df['ReorderQuantity']

    .apply(lambda x: max(x, 0))
)

inventory_df

,Product,CurrentStock,ForecastDemand,ReorderQuantity
0,A,120,180,80
1,B,50,70,40
2,C,200,160,0
3,D,40,90,70


STOCK STATUS

In [4]:
inventory_df['StockStatus'] = (

    inventory_df['ReorderQuantity']

    .apply(
        lambda x:
        'Reorder Needed'
        if x > 0
        else 'Sufficient Stock'
    )
)

inventory_df

,Product,CurrentStock,ForecastDemand,ReorderQuantity,StockStatus
0,A,120,180,80,Reorder Needed
1,B,50,70,40,Reorder Needed
2,C,200,160,0,Sufficient Stock
3,D,40,90,70,Reorder Needed


RISK ANALYSIS

In [5]:
# Days left estimation

inventory_df['DaysLeft'] = (

    inventory_df['CurrentStock']

    / inventory_df['ForecastDemand']

) * 30

# Risk classification

def stock_risk(days):

    if days < 7:
        return 'Critical'

    elif days < 15:
        return 'Medium'

    else:
        return 'Safe'

inventory_df['RiskLevel'] = (

    inventory_df['DaysLeft']

    .apply(stock_risk)
)

inventory_df

,Product,CurrentStock,ForecastDemand,ReorderQuantity,StockStatus,DaysLeft,RiskLevel
0,A,120,180,80,Reorder Needed,20.000000,Safe
1,B,50,70,40,Reorder Needed,21.428571,Safe
2,C,200,160,0,Sufficient Stock,37.500000,Safe
3,D,40,90,70,Reorder Needed,13.333333,Medium


In [6]:
import math

# Example Values

average_daily_demand = 120
lead_time = 7

# Reorder Point

reorder_point = (
    average_daily_demand * lead_time
)

print('Reorder Point:', reorder_point)

# Safety Stock

maximum_daily_usage = 150
maximum_lead_time = 10

safety_stock = (
    (maximum_daily_usage * maximum_lead_time)
    -
    (average_daily_demand * lead_time)
)

print('Safety Stock:', safety_stock)

# EOQ

demand = 5000
ordering_cost = 100
holding_cost = 5

EOQ = math.sqrt(
    (2 * demand * ordering_cost)
    / holding_cost
)

print('Economic Order Quantity:', EOQ)

Reorder Point: 840
Safety Stock: 660
Economic Order Quantity: 447.21359549995793


In [ ]:
inventory_df.to_csv("outputs/inventory.csv", index=False)